In [8]:
%pip install seaborn

  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
Using cached seaborn-0.13.2-py3-none-any.whl (294 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score
import joblib

# Load the dataset
df = pd.read_csv('../data/UCI_Credit_Card.csv')

# Drop unique identifier columns that shouldn't leak into training
if 'ID' in df.columns:
    df = df.drop(columns=['ID'])

# Dynamically intercept variation in default naming conventions
target_col = [col for col in df.columns if col.lower().startswith('default')][0]
df = df.rename(columns={target_col: 'default'})

print(f"✅ Step 1 Complete: Dataset ingested with shape {df.shape}")
print(f"Target column mapped from '{target_col}' -> 'default'")

IndexError: list index out of range

In [10]:
print(df.columns.tolist())

['LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6', 'target']


In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score
import joblib

# Load the dataset
df = pd.read_csv('../data/UCI_Credit_Card.csv')

# Drop unique identifier columns if they are present
if 'ID' in df.columns:
    df = df.drop(columns=['ID'])

# Explicitly map 'target' to 'default'
df = df.rename(columns={'target': 'default'})

print(f"✅ Step 1 Complete: Dataset ingested with shape {df.shape}")
print("Target column 'target' successfully renamed to 'default'")

✅ Step 1 Complete: Dataset ingested with shape (30000, 24)
Target column 'target' successfully renamed to 'default'


In [13]:
# 1. Credit Utilization Rate (Most recent billing statement balance / Total Allowed Line Limit)
df['utilization_rate'] = df['BILL_AMT1'] / (df['LIMIT_BAL'] + 1e-5)
df['utilization_rate'] = df['utilization_rate'].clip(0.0, 1.5)  # Cap extreme over-limit anomalies

# 2. Payment-to-Minimum Ratio (Approximating standard 5% minimum payment hurdle from previous statement)
expected_min_due = df['BILL_AMT2'] * 0.05

# Inline lambda conditional to compute payment coverage safely
df['payment_to_minimum_ratio'] = np.where(
    expected_min_due > 0, 
    df['PAY_AMT1'] / expected_min_due, 
    1.0
)
df['payment_to_minimum_ratio'] = df['payment_to_minimum_ratio'].clip(0.0, 20.0)  # Cap structural outliers

print("✅ Step 2 Complete: Financial ratio profiles successfully calculated.")
print(df[['LIMIT_BAL', 'utilization_rate', 'payment_to_minimum_ratio']].head())

✅ Step 2 Complete: Financial ratio profiles successfully calculated.
   LIMIT_BAL  utilization_rate  payment_to_minimum_ratio
0    20000.0          0.195650                  0.000000
1   120000.0          0.022350                  0.000000
2    90000.0          0.324878                  2.164397
3    50000.0          0.939800                  0.829308
4    50000.0          0.172340                  7.054674


In [14]:
# Isolate high-impact financial features for training
core_features = [
    'LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 
    'PAY_0', 'utilization_rate', 'payment_to_minimum_ratio'
]

X = df[core_features]
y = df['default']

# Split datasets with fixed seed and proportional stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    stratify=y, 
    random_state=42
)

print("✅ Step 3 Complete: Balanced dataset split compiled.")
print(f"Training Matrix Size: {X_train.shape} | Testing Matrix Size: {X_test.shape}")
print(f"Target Base Risk Base Rate: {y_train.mean():.2%}")

✅ Step 3 Complete: Balanced dataset split compiled.
Training Matrix Size: (24000, 8) | Testing Matrix Size: (6000, 8)
Target Base Risk Base Rate: 22.12%


In [15]:
# Compute the exact class weighting multiplier to scale gradient penalties
num_defaults = (y_train == 1).sum()
num_goods = (y_train == 0).sum()
imbalance_multiplier = num_goods / num_defaults

print(f"Applying Imbalance Multiplier: {imbalance_multiplier:.2f}x penalty to missed defaults.\n")

# Configure the XGBoost Classifier
model = XGBClassifier(
    n_estimators=150,
    max_depth=4,
    learning_rate=0.05,
    scale_pos_weight=imbalance_multiplier,  # Forces model to prioritize precision/recall on default class
    random_state=42,
    eval_metric='logloss'
)

print("Training production pipeline weights...")
model.fit(X_train, y_train)

# Generate predictions and raw probability distributions
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print("\n--- Financial Operational Metrics ---")
print(classification_report(y_test, y_pred))
print(f"Engine Separation Capacity (ROC-AUC Score): {roc_auc_score(y_test, y_proba):.4f}")

Applying Imbalance Multiplier: 3.52x penalty to missed defaults.

Training production pipeline weights...

--- Financial Operational Metrics ---
              precision    recall  f1-score   support

           0       0.87      0.81      0.84      4673
           1       0.47      0.59      0.52      1327

    accuracy                           0.76      6000
   macro avg       0.67      0.70      0.68      6000
weighted avg       0.78      0.76      0.77      6000

Engine Separation Capacity (ROC-AUC Score): 0.7680


In [16]:
# Save the model file into your app folder
joblib.dump(model, '../app/model.pkl')
print("✅ Step 5 Complete: Pipeline compiled and exported to ../app/model.pkl")

✅ Step 5 Complete: Pipeline compiled and exported to ../app/model.pkl
